# DOW-30 Walk-Forward Dataset Loader

Test notebook for `load_dataset` and `make_window_data` from `utils.py`.
Windows and paths are defined in `dow_1_train.py`.

In [ ]:
from utils.config import WINDOWS, FULL_DATE, TEST_WINDOWS
from utils.common import load_dataset, make_window_data
import os
from pathlib import Path

MARKET_NAME = "dow"
SUB_EXPERIMENT_DIR = "experiment_1"
MODELS_DIR   = Path(os.environ['FINRL_PATH']) / "models" / MARKET_NAME / SUB_EXPERIMENT_DIR
LOG_DIR      = Path(os.environ['FINRL_PATH']) / "logs"   / MARKET_NAME / SUB_EXPERIMENT_DIR
DATA_DIR  = Path(os.environ["FINRL_PATH"]) / "data" / MARKET_NAME
print(f"DATA_DIR : {DATA_DIR}")
print(f"FULL_DATE: {FULL_DATE}")
print()
print("Windows:")
for w, splits in WINDOWS.items():
    print(f"  {w}  train: {splits['train'][0]} → {splits['train'][1]}  |  test: {splits['test'][0]} → {splits['test'][1]}")

## 1. Load a single dataset (full, unsliced)

In [3]:
# chlovtd (full OHLCV)
df_mvx = load_dataset("mvx")
print(f"mvx (chlovtd): {df_mvx.shape}  cols: {list(df_mvx.columns)}")

# cvtd (no open/high/low)
df_mvx_c = load_dataset("mvx", cvtd=True)
print(f"mvx (cvtd):    {df_mvx_c.shape}  cols: {list(df_mvx_c.columns)}")

df_mvx.head(3)

mvx (chlovtd): (128280, 12)  cols: ['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day', 'close_20_sma', 'close_5_ema', 'close_5_ema_xu_close_20_sma', 'close_5_ema_xd_close_20_sma']
mvx (cvtd):    (128280, 9)  cols: ['date', 'close', 'volume', 'tic', 'day', 'close_20_sma', 'close_5_ema', 'close_5_ema_xu_close_20_sma', 'close_5_ema_xd_close_20_sma']


,date,close,high,low,open,volume,tic,day,close_20_sma,close_5_ema,close_5_ema_xu_close_20_sma,close_5_ema_xd_close_20_sma
0,2009-01-02,2.719142,2.727833,2.551650,2.573223,746015200,AAPL,4,2.719142,2.719142,False,False
1,2009-01-02,39.900166,39.961041,39.061443,39.629609,6547900,AMGN,4,39.900166,39.900166,False,False
2,2009-01-02,2.718000,2.726500,2.553500,2.567500,145928000,AMZN,4,2.718000,2.718000,False,False


## 2. make_window_data — all windows, all splits

In [ ]:
DATASETS = []
    #"flt_mvx2", "flt_mvx2", "mvx2_trg",  "flt_mvx2_trg", #mv2
    #"boll", "bollx", "flt_bollx", "bollx_trg", "flt_bollx_trg" 
#            "mvx2_trg", #"flt_mvx3","flt_mvx3_trg"

#DATASETS = ["all","mv", "mvx", "mvx2", "boll", "bollx", "flt_mvx",  
#            "flt_mvx2", "flt_bollx", "bollmvx", "mvx_trg", 
#            "mvx2_trg", "bollx_trg", "dbmvx", "3mvx1",
#             'mvx2all', '3mvx2',  'mvxall']
#data = make_window_data(DATASETS, windows=["w1", "w2", "w3"], splits=["train", "test"])
data = make_window_data(DATASETS, windows=["w1", "w4", "w3"], splits=["train", "test"])

for w, by_name in data.items():
    for name, by_split in by_name.items():
        for split, entry in by_split.items():
            print(f"  {w}/{name}/{split:5s}: {str(entry['df'].shape):16s}  tag={entry['tag']}")

  w4/mvx2/train: (97560, 12)       tag=dow_chlovtd_mvx2_2009-01-01_to_2021-12-01
  w4/mvx2/test : (7530, 12)        tag=dow_chlovtd_mvx2_2022-01-01_to_2023-01-01


## 3. Selective usage examples

In [4]:
# Train-only for w3 (ready to pass to RL agent)
data_w3_train = make_window_data(["mvx", "bollrsix1"], windows=["w3"], splits=["train"])

df_train = data_w3_train["w3"]["mvx"]["train"]["df"]
tag      = data_w3_train["w3"]["mvx"]["train"]["tag"]
print(f"df shape : {df_train.shape}")
print(f"tag      : {tag}")
print(f"date range: {df_train['date'].min()}  →  {df_train['date'].max()}")
df_train.head(3)

df shape : (120150, 12)
tag      : dow_chlovtd_mvx_2009-01-01_to_2024-12-01
date range: 2009-01-02  →  2024-11-29


,date,close,high,low,open,volume,tic,day,close_20_sma,close_5_ema,close_5_ema_xu_close_20_sma,close_5_ema_xd_close_20_sma
0,2009-01-02,2.719142,2.727833,2.551650,2.573223,746015200,AAPL,4,2.719142,2.719142,False,False
1,2009-01-02,39.900166,39.961041,39.061443,39.629609,6547900,AMGN,4,39.900166,39.900166,False,False
2,2009-01-02,2.718000,2.726500,2.553500,2.567500,145928000,AMZN,4,2.718000,2.718000,False,False


---
## 4. Training Configuration

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from utils.config import ( INDICATORS_ALL,
    INDICATORS_MV, INDICATORS_MV2, INDICATORS_MVX, INDICATORS_MVX2,
    INDICATORS_BOLL, INDICATORS_BOLLX, INDICATORS_BOLLMVX, INDICATORS_BOLLMVX2,
    INDICATORS_3MVX1, INDICATORS_3MVX2, INDICATORS_BOLLRSIX1, INDICATORS_BOLLRSIX2, INDICATORS_DBMVX,
    INDICATORS_MVX2ALL, INDICATORS_MVXALL, INDICATORS_BOLLXALL, INDICATORS_BASE,
    SEEDS)

# ── Parallelism ───────────────────────────────────────────────────────────────
N_WORKERS = 8
GPU_FRAC  = 0.09    # 10 workers × 0.09 ≈ 90% of 4 GB GPU


# ── Algos ─────────────────────────────────────────────────────────────────────
ALGOS = "all"                     # ["a2c","ppo","sac"] or "all"
if ALGOS == "all":
    ALGOS = ["a2c", "ppo", "sac"]

# ── Timesteps per algo ────────────────────────────────────────────────────────
TIMESTEPS = {
    "a2c": 50000,
    "ppo": 50000,
    "sac": 50000,
}

# ── Hyperparameters ───────────────────────────────────────────────────────────
ALGO_PARAMS = {
    "a2c": {"n_steps": 5,    "ent_coef": 0.01, "learning_rate": 0.0007,  "device": "cuda"},
    "ppo": {"n_steps": 2048, "ent_coef": 0.01, "learning_rate": 0.00025,
            "batch_size": 128, "device": "cuda"},
    "sac": {"batch_size": 128, "buffer_size": 100_000, "learning_rate": 0.0001,
            "learning_starts": 100, "ent_coef": "auto_0.1", "device": "cuda"},
}

# ── Indicator lookup ──────────────────────────────────────────────────────────
# Maps base dataset name → indicator column list.
# Handles: "flt_bollx" (strip flt_→ bollx), "_trg" suffix (read bool cols from df).
''' Use this for particular indicator sets needed for training
INDICATORS_BASE = {
    "all":       INDICATORS_ALL,  # all has same indicators as mvx (just different values)
    "mv":        INDICATORS_MV,
    "mv2":       INDICATORS_MV2,
    "mvx":       INDICATORS_MVX,
    "mvx2":      INDICATORS_MVX2,
    "dbmvx":     INDICATORS_DBMVX,  # dbmvx has same indicators as mvx (just different values)
    "boll":      INDICATORS_BOLL,
    "bollx":     INDICATORS_BOLLX,
    "bollmvx":   INDICATORS_BOLLMVX,
    "bollmvx2":  INDICATORS_BOLLMVX2,
    "3mvx1":     INDICATORS_3MVX1,
    "3mvx2":     INDICATORS_3MVX2,
    "bollrsix1": INDICATORS_BOLLRSIX1,
    "bollrsix2": INDICATORS_BOLLRSIX2,
    'mvx2all': INDICATORS_MVX2ALL,
    'mvxall': INDICATORS_MVXALL, 
    'bollxall':INDICATORS_BOLLXALL
}
'''

def get_indicators(dataset_name: str, df=None) -> list:
    """
    Resolve indicator list from dataset name.
      "flt_bollx"  → strip "flt_" → INDICATORS_BOLLX
      "mvx2_trg"   → strip "flt_" if present, ends with "_trg" → bool cols from df
      "mvx"        → INDICATORS_MVX
    cvtd vs chlovtd is encoded in the tag prefix, not the name → no effect here.
    """
    base = dataset_name.removeprefix("flt_")
    if base.endswith("_trg"):
        if df is None:
            raise ValueError(f"df required for _trg dataset '{dataset_name}' to infer indicator columns")
        return list(df.select_dtypes(include=["bool", "boolean"]).columns)
    return INDICATORS_BASE[base]

# ── Datasets and windows to train ─────────────────────────────────────────────
# Default: mv + mvx → 2 datasets × 1 algo × 5 seeds = 10 jobs → fills 10 workers exactly

""" 
DATASETS_TO_TRAIN = [
    "all", 'mvxall', 'bollxall', 'mvx2all',
    "mv", "mvx", "mvx2", "boll", "bollx", "dbmvx", 
    "3mvx1", '3mvx2',  'bollrsix1', 'bollrsix2',"bollmvx", "bollmvx2",
    "flt_mvx", "flt_mvx2", "flt_bollx",  'flt_bollmvx2','flt_3mvx1', 
    'flt_3mvx2','flt_bollrsix1', 'flt_bollrsix2', "flt_dbmvx",
    "mvx_trg", "mvx2_trg", '3mvx1_trg',"bollx_trg", 'bollrsix1_trg', 'bollrsix2_trg',  # ← ADD COMMA HERE
    'flt_bollmvx_trg','flt_bollmvx2_trg','flt_3mvx1_trg', 'flt_3mvx2_trg','flt_bollrsix1_trg', 'flt_bollrsix2_trg', 
    "flt_mvx_trg",  "flt_mvx2_trg",  "flt_dbmvx_trg", "flt_bollx_trg"
]
"""

# Missing # 'bollxall', bollrsix1_trg', 'bollrsix2_trg'
#DATASETS_TO_TRAIN = ["3mvx1", "3mvx2", "bollrsix1", "bollrsix2", "flt_3mvx1", "flt_3mvx2", "flt_bollrsix1", "flt_bollrsix2"]
""" for best results to train with CVDT
DATASETS_TO_TRAIN = [
    "mvx2all",
    "dbmvx",
    "mvxall",
    "bollmvx",
    "flt_mvx2",
    "bollrsix2",
    "flt_bollrsix2",
    "bollx_trg",
    "flt_bollx_trg",
    "flt_dbmvx_trg",
    "flt_3mvx1_trg",
    "flt_bollrsix1_trg",
    "flt_mvx2_trg",
    "flt_bollrsix1",
    "flt_bollx",
    "flt_bollrsix2_trg",
    "flt_bollmvx2_trg",
]"""


# Full experiment (uncomment to run all):
DATASETS_TO_TRAIN = [ "all"]
  #  "flt_mvx2", "flt_mvx2", "mvx2_trg",  "flt_mvx2_trg", #mv2
 #   "boll", "bollx", "flt_bollx", "bollx_trg", "flt_bollx_trg" 
#            "mvx2_trg", #"flt_mvx3","flt_mvx3_trg"
#]

WINDOWS_TO_TRAIN = ["w4"] # for full walk-forward

# ── Summary ───────────────────────────────────────────────────────────────────
total = len(ALGOS) * len(SEEDS) * len(DATASETS_TO_TRAIN) * len(WINDOWS_TO_TRAIN)
print(f"Algos    : {ALGOS}")
print(f"Seeds    : {SEEDS}  ({len(SEEDS)} seeds)")
print(f"Workers  : {N_WORKERS}  |  GPU/worker: {GPU_FRAC*100:.0f}%")
print(f"Datasets : {DATASETS_TO_TRAIN}")
print(f"Windows  : {WINDOWS_TO_TRAIN}")
print(f"Max jobs : {total}  (before resume check)")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"LOG_DIR   : {LOG_DIR}")

## 5. Build Jobs + Resume Check

Scans `MODELS_DIR` for already-trained runs and skips them.
Model naming: `{tag}_{algo}_s{seed}.zip`  
Example: `dow_chlovtd_mvx_2009-01-01_to_2024-11-30_sac_s42.zip`

In [ ]:
from utils.train_workers import train_model_seeded

# Load training slices for selected windows/datasets
data = make_window_data(DATASETS_TO_TRAIN, windows=WINDOWS_TO_TRAIN, cvtd=False, splits=["train"])

all_jobs, skip_count = [], 0

for w, by_name in data.items():
    for name, by_split in by_name.items():
        entry    = by_split["train"]
        tag      = entry["tag"]          # e.g. "dow_chlovtd_mvx_2009-01-01_to_2024-11-30"

        # FinRL StockTradingEnv requires a date-factorized index:
        # df.loc[0] must return ALL tickers for date 0 (not a single row).
        # sort_values + factorize converts the flat row index to date-groups.
        df_train = entry["df"].copy()
        df_train = df_train.sort_values(["date", "tic"], ignore_index=True)
        df_train.index = df_train["date"].factorize()[0]

        indicators = get_indicators(name, df_train)   # handles flt_, _trg, plain

        for algo in ALGOS:
            model_name = f"{tag}_{algo}"
            params     = ALGO_PARAMS[algo]
            steps      = TIMESTEPS[algo]

            for seed in SEEDS:
                run_name = f"{model_name}_s{seed}"
                out_path = MODELS_DIR / f"{run_name}.zip"

                if out_path.exists():
                    skip_count += 1
                    print(f"  ✓ skip  {run_name}")
                    continue

                # positional args match train_model_seeded signature:
                # (df, indicators, algo_name, model_params, timesteps, model_name,
                #  gpu_fraction, seed, policy_kwargs, save_dir, log_dir)
                all_jobs.append((
                    df_train, indicators, algo, params, steps,
                    model_name, GPU_FRAC, seed, None,
                    MODELS_DIR, LOG_DIR,
                ))
# not working bollxall
print(f"\nJobs to run : {len(all_jobs)}")
print(f"Already done: {skip_count}  (skipped)")
if all_jobs:
    print(f"\nFirst job: {all_jobs[0][5]}_s{all_jobs[0][7]}")   # model_name + seed

## 6. Run Training

Dispatches all pending jobs across `N_WORKERS` parallel GPU processes.
Re-run this cell at any time — §5 will have already skipped completed models.

In [5]:
if not all_jobs:
    print("Nothing to train — all models already exist.")
else:
    ctx = multiprocessing.get_context("spawn")
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as executor:
        futures = [executor.submit(train_model_seeded, *j) for j in all_jobs]
        done = 0
        for f in as_completed(futures):
            done += 1
            print(f"  [{done:02d}/{len(all_jobs)}] {f.result()}")
    print("\nTraining complete.")


Stock Dimension: 30, State Space: 301
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007, 'device': 'cuda'}
Using cuda device
Stock Dimension: 30, State Space: 301
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007, 'device': 'cuda'}
Using cuda device
Stock Dimension: 30, State Space: 301
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 128, 'device': 'cuda'}
Using cuda device
Stock Dimension: 30, State Space: 301
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 128, 'device': 'cuda'}
Using cuda device
Stock Dimension: 30, State Space: 301
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007, 'device': 'cuda'}
Using cuda device
Stock Dimension: 30, State Space: 301
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007, 'device': 'cuda'}
Using cuda device
Stock Dimension: 30, State Space: 301
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007, 'device': 'cuda'}
Using cuda device
Stock Dimension: 30, State Spa

/home/main/.conda/envs/finrl/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/home/main/.conda/envs/finrl/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `exp

Logging to /home/main/Programming/finrl/logs/dow/experiment_1/dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s456
[dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s456] training  gpu=9%  seed=456
Logging to /home/main/Programming/finrl/logs/dow/experiment_1/dow_chlovtd_all_2009-01-01_to_2021-12-01_a2c_s123
[dow_chlovtd_all_2009-01-01_to_2021-12-01_a2c_s123] training  gpu=9%  seed=123
Logging to /home/main/Programming/finrl/logs/dow/experiment_1/dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s123
[dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s123] training  gpu=9%  seed=123
Logging to /home/main/Programming/finrl/logs/dow/experiment_1/dow_chlovtd_all_2009-01-01_to_2021-12-01_a2c_s1234
[dow_chlovtd_all_2009-01-01_to_2021-12-01_a2c_s1234] training  gpu=9%  seed=1234


/home/main/.conda/envs/finrl/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Logging to /home/main/Programming/finrl/logs/dow/experiment_1/dow_chlovtd_all_2009-01-01_to_2021-12-01_a2c_s789
[dow_chlovtd_all_2009-01-01_to_2021-12-01_a2c_s789] training  gpu=9%  seed=789


/home/main/.conda/envs/finrl/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


... (5 outputs left out)

/home/main/.conda/envs/finrl/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Logging to /home/main/Programming/finrl/logs/dow/experiment_1/dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s789
[dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s789] training  gpu=9%  seed=789
--------------------------------------
| time/                 |            |
|    fps                | 45         |
|    iterations         | 9400       |
|    time_elapsed       | 1031       |
|    total_timesteps    | 47000      |
| train/                |            |
|    advantage_mean     | 3.04       |
|    advantage_std      | 1.47       |
|    advantage_var      | 2.16       |
|    entropy_loss       | -43.8      |
|    explained_variance | 0.0673     |
|    learning_rate      | 0.0007     |
|    n_updates          | 9399       |
|    policy_loss        | 20.1       |
|    reward             | 0.7497999  |
|    reward_max         | 3.1622105  |
|    reward_mean        | 1.2248254  |
|    reward_min         | -0.6087059 |
|    std                | 1.04       |
|    value_loss         | 2.3

/home/main/.conda/envs/finrl/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Logging to /home/main/Programming/finrl/logs/dow/experiment_1/dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s1234
[dow_chlovtd_all_2009-01-01_to_2021-12-01_ppo_s1234] training  gpu=9%  seed=1234
-----------------------------------------
| time/                   |             |
|    fps                  | 46          |
|    iterations           | 24          |
|    time_elapsed         | 1059        |
|    total_timesteps      | 49152       |
| train/                  |             |
|    advantage_mean       | -0.504      |
|    advantage_std        | 4.69        |
|    advantage_var        | 22          |
|    approx_kl            | 0.023131557 |
|    clip_fraction        | 0.238       |
|    clip_range           | 0.2         |
|    entropy_loss         | -43.6       |
|    explained_variance   | 0.911       |
|    learning_rate        | 0.00025     |
|    loss                 | 5.76        |
|    n_updates            | 230         |
|    policy_gradient_loss | -0.0551     |
|    rew